**Introduction**

In this project, I build a rule-based crypto portfolio using the top 10 non-stable coins by market capitalization and backtest it on 5 years of daily Binance data. The process starts by selecting the large-cap universe from CoinGecko, mapping those assets to Binance spot pairs, and downloading historical prices. Based on this dataset, I apply a systematic portfolio construction approach that combines asset selection, periodic rebalancing, and risk control to improve risk-adjusted returns rather than simply maximizing raw return. The strategy is then evaluated using Sharpe ratio, maximum drawdown, CAGR, and volatility, with BTC buy-and-hold as a benchmark. One important assumption is that the portfolio uses the current top-10 market-cap universe over the full backtest window, which may introduce survivorship bias

In [1]:
!pip install --upgrade binance-historical-data plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.2 MB/s eta 0:00:00


CELL 1 - SETUP & CONFIG

In [2]:
import warnings
warnings.filterwarnings("ignore")

import time
import random
from datetime import timezone

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
np.set_printoptions(suppress=True, precision=6)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

TOP_N = 10
LOOKBACK_YEARS = 5
INTERVAL = "1d"
REBALANCE_FREQ = "W-SUN"

START_DATE = pd.Timestamp.utcnow().tz_localize(None) - pd.DateOffset(years=LOOKBACK_YEARS)
END_DATE = pd.Timestamp.utcnow().tz_localize(None)

TARGET_SHARPE = 1.0
MAX_DRAWDOWN_LIMIT = -0.30

ANNUALIZATION_FACTOR = 365
RISK_FREE_RATE = 0.0

BENCHMARK = "BTCUSDT"
QUOTE_ASSET = "USDT"

EXCLUDED_SYMBOLS = {
    "USDT", "USDC", "BUSD", "DAI", "TUSD", "FDUSD", "USDP", "USDD",
    "WBTC", "WETH", "STETH", "CETH"
}

COINGECKO_BASE = "https://api.coingecko.com/api/v3"

session = requests.Session()
retry = Retry(
    total=5,
    read=5,
    connect=5,
    backoff_factor=1.2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

HEADERS = {
    "accept": "application/json",
    "user-agent": "Mozilla/5.0 (Colab Quant Assignment)"
}

def get_json(url, params=None, sleep=0.2):
    r = session.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    time.sleep(sleep)
    return r.json()

print("Setup done.")
print(f"Backtest window : {START_DATE.date()} -> {END_DATE.date()}")
print(f"Universe        : Top {TOP_N} coins by market cap")
print(f"Rebalance       : {REBALANCE_FREQ}")
print(f"Target          : Sharpe > {TARGET_SHARPE}, MDD >= {MAX_DRAWDOWN_LIMIT:.0%}")

Setup done.
Backtest window : 2021-04-13 -> 2026-04-13
Universe        : Top 10 coins by market cap
Rebalance       : W-SUN
Target          : Sharpe > 1.0, MDD >= -30%


CELL 2 - DATA DOWNLOAD HELPERS

In [3]:

import os
import glob
import warnings
from datetime import datetime, timedelta

import pandas as pd
from binance_historical_data import BinanceDataDumper

warnings.filterwarnings("ignore")

def create_data_dumper(market_type="spot", data_frequency="1d"):
    return BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class=market_type,   # "spot", "um", "cm"
        data_type="klines",
        data_frequency=data_frequency,
    )

def get_list_all_trading_pairs(market_type="spot", data_frequency="1d"):
    dumper = create_data_dumper(market_type=market_type, data_frequency=data_frequency)
    pairs = dumper.get_list_all_trading_pairs()
    return sorted(list(set(pairs)))

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [
        t for t in tickers
        if t.endswith("USDT") and not any(x in t for x in exclude_keywords)
    ]

def get_min_start_date_for_ticker(ticker, market_type="spot", data_frequency="1d"):
    dumper = create_data_dumper(market_type=market_type, data_frequency=data_frequency)
    return dumper.get_min_start_date_for_ticker(ticker)

def download_ticker_data(
    ticker,
    date_start,
    date_end,
    market_type="spot",
    data_frequency="1d",
    is_to_update_existing=False
):
    dumper = create_data_dumper(market_type=market_type, data_frequency=data_frequency)

    if isinstance(date_start, str):
        date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    if isinstance(date_end, str):
        date_end = datetime.strptime(date_end, "%Y-%m-%d").date()

    dumper.dump_data(
        tickers=[ticker],
        date_start=date_start,
        date_end=date_end,
        is_to_update_existing=is_to_update_existing,
    )

def detect_timestamp_unit(timestamp):
    s = str(int(timestamp))
    if len(s) == 13:
        return "ms"
    elif len(s) == 16:
        return "us"
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors="coerce")

def read_csv_file(file_path):
    df = pd.read_csv(file_path, header=None)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df["open_time"] = df["open_time"].apply(convert_timestamp)
    df["close_time"] = df["close_time"].apply(convert_timestamp)

    numeric_cols = ["Open", "High", "Low", "Close", "volume"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

def process_csv_files(ticker, data_frequency="1d", market_type="spot"):
    base_path = market_type

    daily_path = os.path.join(os.getcwd(), f"{base_path}/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"{base_path}/monthly/klines/{ticker}/{data_frequency}")

    daily_files = glob.glob(os.path.join(daily_path, "*.csv"))
    monthly_files = glob.glob(os.path.join(monthly_path, "*.csv"))

    all_files = sorted(monthly_files) + sorted(daily_files)

    if not all_files:
        print(f"Không có file CSV cho {ticker}")
        return None

    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data = data.sort_values("open_time").drop_duplicates(subset=["open_time"]).reset_index(drop=True)

    return data

print("Helper functions loaded.")

Helper functions loaded.


CELL 3 - UNIVERSE SELECTION

In [4]:
def get_top_coins_by_market_cap(top_n=10, excluded_symbols=None):
    if excluded_symbols is None:
        excluded_symbols = set()

    url = f"{COINGECKO_BASE}/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 50,
        "page": 1,
        "sparkline": "false"
    }

    data = get_json(url, params=params, sleep=0.5)
    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError("CoinGecko trả về rỗng.")

    df["symbol"] = df["symbol"].str.upper()
    df = df[~df["symbol"].isin(excluded_symbols)].copy()

    keep_cols = ["id", "symbol", "name", "market_cap", "market_cap_rank", "current_price", "total_volume"]
    df = df[keep_cols].sort_values(["market_cap_rank", "market_cap"], ascending=[True, False])

    return df.head(top_n).reset_index(drop=True)

def build_binance_pair_map(market_type="spot", data_frequency="1d"):
    all_pairs = get_list_all_trading_pairs(market_type=market_type, data_frequency=data_frequency)
    usdt_pairs = filter_usdt_tickers(all_pairs)

    pair_df = pd.DataFrame({"binance_symbol": usdt_pairs})
    pair_df["baseAsset"] = pair_df["binance_symbol"].str.replace("USDT", "", regex=False)

    return pair_df

def map_top_coins_to_binance(top_coins_df, binance_pair_df):
    df = top_coins_df.copy()
    df["baseAsset"] = df["symbol"].str.upper()

    mapped = df.merge(binance_pair_df, on="baseAsset", how="left")


    manual_map = {
        "DOGE": "DOGEUSDT",
        "TON": "TONUSDT",
        "ADA": "ADAUSDT",
        "SOL": "SOLUSDT",
        "XRP": "XRPUSDT",
        "TRX": "TRXUSDT",
        "BNB": "BNBUSDT",
        "BTC": "BTCUSDT",
        "ETH": "ETHUSDT"
    }

    mapped["binance_symbol"] = mapped.apply(
        lambda row: manual_map.get(row["baseAsset"], row["binance_symbol"]),
        axis=1
    )

    return mapped


top_coins = get_top_coins_by_market_cap(
    top_n=TOP_N,
    excluded_symbols=EXCLUDED_SYMBOLS
)

binance_pair_map = build_binance_pair_map(
    market_type="spot",
    data_frequency=INTERVAL
)

universe = map_top_coins_to_binance(top_coins, binance_pair_map)

print("Top coins by market cap:")
display(top_coins)

print("\nMapped Binance universe:")
display(universe)

missing = universe[universe["binance_symbol"].isna()]
if len(missing) > 0:
    print("Coin chưa map được sang Binance:")
    display(missing)
else:
    print("Map sang Binance pair xong.")

Top coins by market cap:


,id,symbol,name,market_cap,market_cap_rank,current_price,total_volume
0,bitcoin,BTC,Bitcoin,1415665606321,1,70723.000000,2.816574e+10
1,ethereum,ETH,Ethereum,263783425212,2,2184.880000,1.305581e+10
2,ripple,XRP,XRP,81352242371,4,1.320000,1.675014e+09
3,binancecoin,BNB,BNB,81262754546,5,595.950000,8.511661e+08
4,solana,SOL,Solana,47059440789,7,81.910000,2.589614e+09
5,tron,TRX,TRON,30520399255,8,0.322034,5.157519e+08
6,figure-heloc,FIGR_HELOC,Figure Heloc,17311391077,9,1.040000,1.029950e+05
7,dogecoin,DOGE,Dogecoin,13960561763,10,0.090762,8.633597e+08
8,usds,USDS,USDS,11500113792,11,0.999817,5.162709e+07
9,whitebit,WBT,WhiteBIT Coin,11085928559,12,51.940000,3.940737e+07



Mapped Binance universe:


,id,symbol,name,market_cap,market_cap_rank,current_price,total_volume,baseAsset,binance_symbol
0,bitcoin,BTC,Bitcoin,1415665606321,1,70723.000000,2.816574e+10,BTC,BTCUSDT
1,ethereum,ETH,Ethereum,263783425212,2,2184.880000,1.305581e+10,ETH,ETHUSDT
2,ripple,XRP,XRP,81352242371,4,1.320000,1.675014e+09,XRP,XRPUSDT
3,binancecoin,BNB,BNB,81262754546,5,595.950000,8.511661e+08,BNB,BNBUSDT
4,solana,SOL,Solana,47059440789,7,81.910000,2.589614e+09,SOL,SOLUSDT
5,tron,TRX,TRON,30520399255,8,0.322034,5.157519e+08,TRX,TRXUSDT
6,figure-heloc,FIGR_HELOC,Figure Heloc,17311391077,9,1.040000,1.029950e+05,FIGR_HELOC,NaN
7,dogecoin,DOGE,Dogecoin,13960561763,10,0.090762,8.633597e+08,DOGE,DOGEUSDT
8,usds,USDS,USDS,11500113792,11,0.999817,5.162709e+07,USDS,NaN
9,whitebit,WBT,WhiteBIT Coin,11085928559,12,51.940000,3.940737e+07,WBT,NaN


Coin chưa map được sang Binance:


,id,symbol,name,market_cap,market_cap_rank,current_price,total_volume,baseAsset,binance_symbol
6,figure-heloc,FIGR_HELOC,Figure Heloc,17311391077,9,1.040000,102995.0,FIGR_HELOC,NaN
8,usds,USDS,USDS,11500113792,11,0.999817,51627087.0,USDS,NaN
9,whitebit,WBT,WhiteBIT Coin,11085928559,12,51.940000,39407370.0,WBT,NaN


CELL 4 - DATA DOWNLOAD

In [5]:
date_start = START_DATE.strftime("%Y-%m-%d")
date_end = END_DATE.strftime("%Y-%m-%d")

valid_universe = universe.dropna(subset=["binance_symbol"]).copy()
symbols_to_download = valid_universe["binance_symbol"].unique().tolist()

if BENCHMARK not in symbols_to_download:
    symbols_to_download.append(BENCHMARK)

print("Symbols to download:", symbols_to_download)
print("Date range:", date_start, "->", date_end)

for ticker in symbols_to_download:
    try:
        print(f"Downloading {ticker} ...")
        download_ticker_data(
            ticker=ticker,
            date_start=date_start,
            date_end=date_end,
            market_type="spot",
            data_frequency=INTERVAL,
            is_to_update_existing=False
        )
    except Exception as e:
        print(f"{ticker}: {e}")

print("Download completed.")

Symbols to download: ['BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'BNBUSDT', 'SOLUSDT', 'TRXUSDT', 'DOGEUSDT']
Date range: 2021-04-13 -> 2026-04-13
---> Found overall tickers: 618
---> Filter to asked tickers: 1
------> Tickers left: 1
Download full data for 1 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20210413
---> End Date: 20260412


Tickers:   0%|          | 0/1 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/60 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/12 [00:00<?, ?files/s]

Tried to dump data for 1 tickers:
---> For BTCUSDT new data saved for: 60 months 12 days
---> Found overall tickers: 618
---> Filter to asked tickers: 1
------> Tickers left: 1
Download full data for 1 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20210413
---> End Date: 20260412


...

Tried to dump data for 1 tickers:
---> For TRXUSDT new data saved for: 59 months 12 days
---> Found overall tickers: 618
---> Filter to asked tickers: 1
------> Tickers left: 1
Download full data for 1 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20210413
---> End Date: 20260412


Tickers:   0%|          | 0/1 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/60 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/12 [00:00<?, ?files/s]

Tried to dump data for 1 tickers:
---> For DOGEUSDT new data saved for: 60 months 12 days
Download completed.


CELL 5 - BUILD PRICE MATRIX

In [6]:
price_dict = {}

for ticker in symbols_to_download:
    df = process_csv_files(
        ticker=ticker,
        data_frequency=INTERVAL,
        market_type="spot"
    )

    if df is None or df.empty:
        print(f"⚠️ Không có dữ liệu cho {ticker}")
        continue

    tmp = df[["open_time", "Close"]].copy()
    tmp = tmp.rename(columns={"open_time": "date", "Close": ticker})
    tmp = tmp.dropna()
    tmp["date"] = pd.to_datetime(tmp["date"]).dt.floor("D")
    tmp = tmp.drop_duplicates(subset=["date"]).set_index("date")

    price_dict[ticker] = tmp[ticker]

prices = pd.concat(price_dict.values(), axis=1)
prices.columns = list(price_dict.keys())
prices = prices.sort_index()

prices = prices.loc[(prices.index >= pd.to_datetime(date_start)) & (prices.index <= pd.to_datetime(date_end))]

min_obs = int(len(prices.index.unique()) * 0.8)
prices = prices.dropna(axis=1, thresh=min_obs)
prices_raw = prices.copy()
prices = prices.ffill()

print("Price matrix shape:", prices.shape)
display(prices.head())
display(prices.tail())

Price matrix shape: (1826, 7)


,BTCUSDT,ETHUSDT,XRPUSDT,BNBUSDT,SOLUSDT,TRXUSDT,DOGEUSDT
date,,,,,,,
2021-04-13,63575.00,2299.19,1.79611,551.6442,27.4507,0.14622,0.094402
2021-04-14,62959.53,2432.60,1.83468,550.0182,26.4067,0.14040,0.121433
2021-04-15,63159.98,2514.04,1.75781,541.8824,27.7816,0.16421,0.182360
2021-04-16,61334.80,2422.67,1.54805,508.2743,25.2996,0.16091,0.365443
2021-04-17,60006.66,2317.60,1.53896,514.6861,24.8500,0.15535,0.282253


,BTCUSDT,ETHUSDT,XRPUSDT,BNBUSDT,SOLUSDT,TRXUSDT,DOGEUSDT
date,,,,,,,
2026-04-08,71069.93,2190.00,1.3428,601.16,82.57,0.3180,0.09240
2026-04-09,71787.97,2190.20,1.3438,603.03,83.33,0.3203,0.09260
2026-04-10,72962.70,2245.05,1.3559,605.54,84.83,0.3192,0.09368
2026-04-11,73043.16,2284.99,1.3546,607.05,84.93,0.3189,0.09309
2026-04-12,70740.98,2191.65,1.3247,592.19,81.53,0.3213,0.09084


CELL 6 - DATA QUALITY CHECK

In [7]:
coverage = prices_raw.notna().sum().sort_values(ascending=False).to_frame("obs")
coverage["coverage_pct"] = coverage["obs"] / len(prices_raw)

print("Data coverage by symbol:")
display(coverage)

usable_cols = coverage[coverage["coverage_pct"] >= 0.80].index.tolist()
prices = prices[usable_cols].copy()

print("Usable symbols:", usable_cols)

if BENCHMARK not in prices.columns:
    raise ValueError(f"Benchmark {BENCHMARK} không có trong price matrix.")

Data coverage by symbol:


,obs,coverage_pct
BTCUSDT,1826,1.000000
ETHUSDT,1826,1.000000
XRPUSDT,1826,1.000000
BNBUSDT,1826,1.000000
SOLUSDT,1826,1.000000
DOGEUSDT,1826,1.000000
TRXUSDT,1796,0.983571


Usable symbols: ['BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'BNBUSDT', 'SOLUSDT', 'DOGEUSDT', 'TRXUSDT']


CELL 7 - RETURN CALCULATION

In [8]:
returns = prices.pct_change().replace([np.inf, -np.inf], np.nan)
returns = returns.dropna(how="all")

benchmark_returns = returns[BENCHMARK].dropna()

print("Returns shape:", returns.shape)
display(returns.head())

Returns shape: (1825, 7)


,BTCUSDT,ETHUSDT,XRPUSDT,BNBUSDT,SOLUSDT,DOGEUSDT,TRXUSDT
date,,,,,,,
2021-04-14,-0.009681,0.058025,0.021474,-0.002948,-0.038032,0.286338,-0.039803
2021-04-15,0.003184,0.033479,-0.041898,-0.014792,0.052066,0.501734,0.169587
2021-04-16,-0.028898,-0.036344,-0.119330,-0.062021,-0.089340,1.003968,-0.020096
2021-04-17,-0.021654,-0.043370,-0.005872,0.012615,-0.017771,-0.227642,-0.034553
2021-04-18,-0.064270,-0.035364,-0.085116,-0.064601,0.301239,0.141988,-0.079369


CELL 8 - STRATEGY DEFINITION

In [9]:
LOOKBACK_MOM = 90
LOOKBACK_VOL = 30
TOP_K = 3
MAX_WEIGHT = 0.50
TARGET_VOL_ANNUAL = 0.18
MA_FILTER_DAYS = 200

def get_rebalance_dates(price_df, freq="W-SUN"):
    return price_df.resample(freq).last().index

def calc_momentum(price_window, lookback=90):
    return price_window.iloc[-1] / price_window.iloc[-lookback] - 1

def calc_inverse_vol_weights(ret_window, max_weight=0.50):
    vol = ret_window.std()
    inv_vol = 1 / vol.replace(0, np.nan)
    w = inv_vol / inv_vol.sum()
    w = w.clip(upper=max_weight)

    if w.sum() == 0:
        w = pd.Series(1 / len(ret_window.columns), index=ret_window.columns)
    else:
        w = w / w.sum()

    return w

def calc_realized_vol(ret_window, annual_factor=365):
    if ret_window.empty:
        return np.nan
    port_vol = ret_window.std().mean() * np.sqrt(annual_factor)
    return port_vol

def calc_portfolio_scaler(ret_window, target_vol_annual=0.18, annual_factor=365):
    if ret_window.empty:
        return 0.0

    avg_daily_vol = ret_window.std().mean()
    realized_vol_annual = avg_daily_vol * np.sqrt(annual_factor)

    if pd.isna(realized_vol_annual) or realized_vol_annual == 0:
        return 0.0

    scaler = target_vol_annual / realized_vol_annual
    scaler = min(max(scaler, 0.0), 1.0)  # không leverage, chỉ giảm exposure
    return scaler

CELL 9 - BACKTEST ENGINE

In [10]:
rebalance_dates = get_rebalance_dates(prices, REBALANCE_FREQ)
portfolio_returns = pd.Series(index=returns.index, dtype=float)
weights_history = {}

for i, reb_date in enumerate(rebalance_dates):
    hist_prices = prices.loc[:reb_date].dropna(axis=1, how="all")
    hist_returns = returns.loc[:reb_date].dropna(axis=1, how="all")

    if len(hist_prices) < max(LOOKBACK_MOM, LOOKBACK_COV) + 5:
        continue

    tradable_cols = [c for c in hist_prices.columns if c != BENCHMARK]

    if len(tradable_cols) < TOP_K:
        continue

    px = hist_prices[tradable_cols].copy()

    px = px.dropna(axis=1, thresh=max(LOOKBACK_MOM, LOOKBACK_COV))
    if px.shape[1] < TOP_K:
        continue

    mom = compute_momentum(px.dropna(), lookback=LOOKBACK_MOM).sort_values(ascending=False)
    selected = mom.head(TOP_K).index.tolist()

    ret_window = returns[selected].dropna().tail(LOOKBACK_COV)
    if ret_window.shape[0] < 20:
        continue

    cov = ret_window.cov()
    w = min_variance_weights(cov, max_weight=MAX_WEIGHT, min_weight=MIN_WEIGHT)
    weights_history[reb_date] = w

    if i < len(rebalance_dates) - 1:
        next_reb_date = rebalance_dates[i + 1]
        hold_period = returns.loc[(returns.index > reb_date) & (returns.index <= next_reb_date), selected]
    else:
        hold_period = returns.loc[returns.index > reb_date, selected]

    if hold_period.empty:
        continue

    pr = hold_period.mul(w, axis=1).sum(axis=1)
    portfolio_returns.loc[pr.index] = pr

portfolio_returns = portfolio_returns.dropna()

print("Portfolio returns length:", len(portfolio_returns))
print("Number of rebalances:", len(weights_history))
display(pd.DataFrame(weights_history).T.head())

NameError: name 'LOOKBACK_COV' is not defined

CELL 10 - PERFORMANCE METRICS

In [ ]:
def calc_sharpe(ret, annual_factor=365, rf=0.0):
    ret = ret.dropna()
    if len(ret) == 0 or ret.std() == 0:
        return np.nan
    return (ret.mean() * annual_factor - rf) / (ret.std() * np.sqrt(annual_factor))

def calc_mdd(ret):
    equity = (1 + ret.fillna(0)).cumprod()
    peak = equity.cummax()
    dd = equity / peak - 1
    return dd.min()

def calc_cagr(ret, annual_factor=365):
    ret = ret.dropna()
    if len(ret) == 0:
        return np.nan
    equity = (1 + ret).cumprod()
    years = len(ret) / annual_factor
    if years <= 0:
        return np.nan
    return equity.iloc[-1] ** (1 / years) - 1

common_index = portfolio_returns.index.intersection(benchmark_returns.index)
port_eval = portfolio_returns.loc[common_index]
bench_eval = benchmark_returns.loc[common_index]

results = pd.DataFrame({
    "Strategy": {
        "CAGR": calc_cagr(port_eval),
        "Volatility": port_eval.std() * np.sqrt(365),
        "Sharpe": calc_sharpe(port_eval),
        "MDD": calc_mdd(port_eval)
    },
    "Benchmark_BTC": {
        "CAGR": calc_cagr(bench_eval),
        "Volatility": bench_eval.std() * np.sqrt(365),
        "Sharpe": calc_sharpe(bench_eval),
        "MDD": calc_mdd(bench_eval)
    }
})

print("Performance summary:")
display(results)

CELL 11 - VISUALIZATION

In [ ]:
strategy_equity = (1 + port_eval).cumprod()
benchmark_equity = (1 + bench_eval).cumprod()

strategy_dd = strategy_equity / strategy_equity.cummax() - 1
benchmark_dd = benchmark_equity / benchmark_equity.cummax() - 1

plt.figure(figsize=(14, 5))
plt.plot(strategy_equity.index, strategy_equity, label="Strategy")
plt.plot(benchmark_equity.index, benchmark_equity, label="BTC Benchmark")
plt.title("Equity Curve")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(strategy_dd.index, strategy_dd, label="Strategy Drawdown")
plt.plot(benchmark_dd.index, benchmark_dd, label="BTC Drawdown")
plt.axhline(-0.30, linestyle="--")
plt.title("Drawdown")
plt.legend()
plt.grid(True)
plt.show()

CELL 12 - FINAL EVALUATION

In [ ]:
final_sharpe = results.loc["Sharpe", "Strategy"]
final_mdd = results.loc["MDD", "Strategy"]

print("Final Strategy Sharpe:", round(final_sharpe, 4))
print("Final Strategy MDD   :", round(final_mdd, 4))

if (final_sharpe > TARGET_SHARPE) and (final_mdd >= MAX_DRAWDOWN_LIMIT):
    print("Strategy meets the target: Sharpe > 1 and MDD not worse than -30%")
else:
    print("Strategy does NOT meet the target yet")